# CNN 1D — Clasificación de Etapas de Sueño
**Proyecto 3 — MAIA Grupo 25 — Omar Diaz**

### Flujo:
```
S3 Diego (dvc pull) → Colab GPU → modelo entrenado → S3 + Drive
```

### Antes de ejecutar:
1. Activar GPU A100: `Entorno de ejecución → Cambiar tipo → A100`
2. Agregar secretos AWS en el panel 🔑 Secrets:
   - `AWS_ACCESS_KEY_ID`
   - `AWS_SECRET_ACCESS_KEY`
   - `AWS_SESSION_TOKEN`

⚠️ Las credenciales de AWS Academy expiran cada 4h — actualiza los secretos si dan error.

## 1. Verificar GPU y disco disponible

In [ ]:
import torch, shutil

print('=== GPU ===')
print(f'Disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Modelo:  {torch.cuda.get_device_name(0)}')
    print(f'Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ Sin GPU — activa A100 en Entorno de ejecución')

print('\n=== DISCO ===')
total, used, free = shutil.disk_usage('/')
print(f'Total: {total/1e9:.0f} GB')
print(f'Libre: {free/1e9:.0f} GB')
if free/1e9 < 18:
    print('⚠️ Poco espacio — necesitas al menos 18 GB libres')
else:
    print('✓ Espacio suficiente')

## 2. Instalar dependencias

In [ ]:
!pip install 'dvc[s3]' mlflow scikit-learn awscli -q
print('✓ Dependencias instaladas')

## 3. Cargar credenciales AWS desde Colab Secrets

In [ ]:
from google.colab import userdata
import os

# Leer desde Colab Secrets (panel 🔑 izquierdo)
# Nunca hardcodear credenciales en el código
os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_SESSION_TOKEN']     = userdata.get('AWS_SESSION_TOKEN')
os.environ['AWS_DEFAULT_REGION']    = 'us-east-2'

# Verificar sin exponer valores
print('✓ Credenciales cargadas desde Colab Secrets')
print(f'  Region: {os.environ["AWS_DEFAULT_REGION"]}')
print(f'  Key ID: {os.environ["AWS_ACCESS_KEY_ID"][:8]}...')

# Verificar acceso al bucket de Diego
print('\nVerificando acceso a S3...')
!aws s3 ls s3://sleep-cnn-grupo25/
print('✓ Acceso confirmado')

## 4. Clonar repo de Diego y bajar datos con DVC
DVC traduce los `.dvc` files → descarga los `.npy` reales desde S3.

⚠️ Esto tarda ~10-15 min (16 GB desde S3). Solo se hace una vez por sesión.

In [ ]:
import os

# Clonar repo de Diego (tiene los .dvc files)
if not os.path.exists('/content/repo-diego'):
    !git clone https://github.com/amquinteroc1/MAIA-Grupo25-Proyecto.git /content/repo-diego

%cd /content/repo-diego
!git checkout feature/diego-cnn-loader
!git pull

print('\nArchivos .dvc encontrados:')
!ls data_processed/*.dvc

In [ ]:
import time
%cd /content/repo-diego

# Configurar DVC remote (bucket de Diego)
!dvc remote add -d cnn-remote s3://sleep-cnn-grupo25 --force
!dvc remote modify cnn-remote region us-east-2

# Bajar datos
print('Bajando datos con DVC desde S3...')
t0 = time.time()
!dvc pull
print(f'\n✓ Datos descargados en {(time.time()-t0)/60:.1f} min')

# Verificar
!ls -lh data_processed/*.npy

## 5. Verificar datos — test de Diego (10/10 checks)

In [ ]:
%cd /content/repo-diego
!python src/data/test_data_cnn.py

## 6. Clonar repo de Omar y enlazar datos
No copiamos los 16 GB — usamos un symlink para que el código de Omar
vea los datos que ya están en `/content/repo-diego/data_processed/`.

In [ ]:
import os, shutil

# Clonar rama de Omar
if not os.path.exists('/content/repo-omar'):
    !git clone https://github.com/amquinteroc1/MAIA-Grupo25-Proyecto.git /content/repo-omar

%cd /content/repo-omar
!git checkout feature/omar-cnn1d
!git pull

# Symlink: data_processed → datos de Diego (sin copiar)
LOCAL_DATA = '/content/repo-omar/data_processed'
DIEGO_DATA = '/content/repo-diego/data_processed'

if os.path.exists(LOCAL_DATA) and not os.path.islink(LOCAL_DATA):
    shutil.rmtree(LOCAL_DATA)
if not os.path.exists(LOCAL_DATA):
    os.symlink(DIEGO_DATA, LOCAL_DATA)

print('✓ Repo de Omar listo')
print(f'✓ Symlink: {LOCAL_DATA} → {DIEGO_DATA}')
!ls /content/repo-omar/data_processed/*.npy | head -6

## 7. Verificar arquitectura CNN 1D

In [ ]:
%cd /content/repo-omar
!python src/models/cnn1d.py

## 8. Montar Drive (para guardar modelo al final)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_MODELS = '/content/drive/MyDrive/MAIA_Proyecto3/models'
os.makedirs(DRIVE_MODELS, exist_ok=True)
print(f'✓ Drive listo: {DRIVE_MODELS}')

## 9. Entrenamiento de prueba — 5 épocas para validar pipeline

In [ ]:
%cd /content/repo-omar/src/models
!python train_cnn.py --epochs 5 --batch_size 256 --lr 1e-3
# batch_size 256 aprovecha mejor la A100

## 10. Entrenamiento completo — 30 épocas

In [ ]:
%cd /content/repo-omar/src/models
!python train_cnn.py \
    --epochs 30 \
    --batch_size 256 \
    --lr 1e-3 \
    --dropout 0.5

## 11. Ver resultados MLflow

In [ ]:
import mlflow

mlflow.set_experiment('sleep-stage-cnn1d')
runs = mlflow.search_runs()
cols = [
    'run_id',
    'metrics.best_val_f1_macro',
    'metrics.best_val_f1_n1',
    'metrics.best_val_f1_rem',
    'params.epochs',
    'params.batch_size',
    'params.lr'
]
print(runs[[c for c in cols if c in runs.columns]].to_string())

## 12. Guardar modelo en Drive y subir a S3

In [ ]:
import shutil, os

MODEL_SRC    = '/content/repo-omar/experiments/cnn1d/best_cnn1d.pt'
DRIVE_MODELS = '/content/drive/MyDrive/MAIA_Proyecto3/models'
MODEL_DST    = f'{DRIVE_MODELS}/best_cnn1d.pt'

if os.path.exists(MODEL_SRC):
    size_mb = os.path.getsize(MODEL_SRC) / 1e6
    print(f'Modelo: {size_mb:.1f} MB')

    # 1. Drive (permanente — para próximas sesiones)
    shutil.copy(MODEL_SRC, MODEL_DST)
    print(f'✓ Guardado en Drive: {MODEL_DST}')

    # 2. S3 (para que Andrés lo use en la API)
    S3_MODEL = 's3://sleep-cnn-grupo25/models/best_cnn1d.pt'
    !aws s3 cp {MODEL_SRC} {S3_MODEL}
    print(f'✓ Subido a S3: {S3_MODEL}')
else:
    print(f'ERROR: No se encontró {MODEL_SRC}')

## 13. Bajar modelo a local y hacer push al repo

Desde tu WSL local ejecuta:
```bash
cd MAIA-Grupo25-Proyecto
git checkout feature/omar-cnn1d

# Configurar credenciales
aws configure

# Bajar modelo desde S3
mkdir -p experiments/cnn1d
aws s3 cp s3://sleep-cnn-grupo25/models/best_cnn1d.pt \
    experiments/cnn1d/best_cnn1d.pt

# Push al repo
git add experiments/cnn1d/best_cnn1d.pt
git commit -m 'feat: modelo CNN1D entrenado 30 epochs'
git push origin feature/omar-cnn1d
```